# Workflow Orchestration: Apache Airflow & Nextflow
### A Simple Step-by-Step Tutorial with Real Examples

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## What are workflow orchestrators and why do you need one?

When your analysis grows beyond a single script, you need a tool that:

- Runs steps **in the right order** (respects dependencies)
- **Retries** failed steps automatically
- Runs independent steps **in parallel**
- Gives you a **visual dashboard** of what ran and what failed
- Makes your pipeline **reproducible** and shareable

```
Without orchestration        With orchestration
--------------------         ------------------
run_step1.sh                 One DAG / workflow file
run_step2.sh                 Visual monitoring dashboard
run_step3.sh                 Auto-retry on failure
...forgot step 2 failed      Parallel execution
```

## Airflow vs Nextflow: choosing the right tool

| Feature | Apache Airflow | Nextflow |
|---------|---------------|----------|
| Best for | Data pipelines, ETL, scheduling | Scientific/bioinformatics pipelines |
| Language | Python (DAGs) | Groovy/DSL2 (processes + channels) |
| Scheduling | Yes (cron-based) | No (trigger-based) |
| Containerisation | Optional | First-class (Docker/Singularity) |
| HPC/Cloud | Via operators | Native (SLURM, AWS, GCP, Azure) |
| Learning curve | Medium | Low for scientists |
| Use in toxicology | ETL, report generation | ADMET pipelines, screening |

## What this tutorial covers

| Part | Tool | What you build |
|------|------|----------------|
| A1 | Airflow | Install and run your first DAG |
| A2 | Airflow | Toxicology data pipeline (PubChem fetch + ADMET) |
| A3 | Airflow | Scheduling, sensors, and email alerts |
| B1 | Nextflow | Install and run your first workflow |
| B2 | Nextflow | Molecular docking screening pipeline |
| B3 | Nextflow | Docker containers + HPC submission |

In [ ]:
# ── Complete pip-only environment setup for this tutorial ────────────────────
# Run these commands in your terminal BEFORE opening this notebook.

setup_commands = '''
# ── AIRFLOW environment ─────────────────────────────────────────────────────
python -m venv ~/envs/airflow
source ~/envs/airflow/bin/activate        # macOS/Linux
# .\\envs\\airflow\\Scripts\\activate      # Windows

pip install --upgrade pip

pip install apache-airflow==2.9.1 \\
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.1/constraints-3.10.txt"

# Airflow providers (optional extras)
pip install apache-airflow-providers-http
pip install apache-airflow-providers-slack
pip install apache-airflow-providers-docker

# Scientific packages used in the toxicology DAG
pip install rdkit requests pandas numpy matplotlib

# Register as VS Code / Jupyter kernel
pip install ipykernel
python -m ipykernel install --user --name=airflow --display-name="Python (airflow)"

# Initialise Airflow
airflow db init
airflow users create --username admin --password admin \\
  --firstname Admin --lastname User --role Admin --email admin@example.com

# ── NEXTFLOW environment ────────────────────────────────────────────────────
# Nextflow is a Java application -- install via official curl installer
# (not pip; there is no pip wheel for Nextflow)
java -version                               # needs Java 11+
curl -s https://get.nextflow.io | bash
mv nextflow ~/bin/ && chmod +x ~/bin/nextflow
nextflow -version                           # verify

# Python packages used inside Nextflow processes
python -m venv ~/envs/nextflow
source ~/envs/nextflow/bin/activate
pip install --upgrade pip
pip install rdkit meeko numpy pandas matplotlib scipy
pip install ipykernel
python -m ipykernel install --user --name=nextflow --display-name="Python (nextflow)"
'''

print(setup_commands)

# ── Verify what is already installed in this kernel ─────────────────────────
import sys, subprocess

def check(name, import_name=None):
    try:
        mod = __import__(import_name or name)
        ver = getattr(mod, '__version__', 'installed')
        return f'OK  ({ver})'
    except ImportError:
        return 'MISSING -- pip install ' + name

def check_cmd(cmd):
    try:
        r = subprocess.run([cmd, '--version'], capture_output=True, text=True, timeout=5)
        line = (r.stdout + r.stderr).strip().split('\n')[0][:50]
        return f'OK  -- {line}'
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return 'NOT FOUND'

print('Current environment check:')
for name, imp in [('airflow','airflow'), ('rdkit','rdkit'),
                   ('requests','requests'), ('pandas','pandas'),
                   ('numpy','numpy'), ('matplotlib','matplotlib')]:
    print(f'  {name:12s}: {check(name, imp)}')

print()
for cmd in ['airflow', 'nextflow', 'java']:
    print(f'  {cmd:12s}: {check_cmd(cmd)}')


---
# PART A — Apache Airflow

## A1: Installation and Your First DAG

### Core concepts in 60 seconds

```
DAG  (Directed Acyclic Graph)
  = the whole pipeline definition
  = a Python file in ~/airflow/dags/

Task
  = one step in the pipeline
  = a Python function decorated with @task

Operator
  = pre-built task type (PythonOperator, BashOperator, etc.)

Dependency
  = task_a >> task_b  means 'run task_b after task_a'

DAG Run
  = one execution of the whole pipeline
```

### Installation (pip only)

```bash
# Step 1: Create a virtual environment
python -m venv ~/envs/airflow
source ~/envs/airflow/bin/activate       # macOS / Linux
# .\envs\airflow\Scripts\activate        # Windows

# Step 2: Upgrade pip first (important)
pip install --upgrade pip

# Step 3: Install Airflow
# The constraint file pins all sub-dependencies to known-good versions
AIRFLOW_VERSION=2.9.1
PYTHON_VERSION=$(python -c 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")')

pip install "apache-airflow==${AIRFLOW_VERSION}" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-${AIRFLOW_VERSION}/constraints-${PYTHON_VERSION}.txt"

# Step 4: Install extra providers you need
pip install apache-airflow-providers-http      # HttpSensor
pip install apache-airflow-providers-slack     # SlackWebhookOperator
pip install apache-airflow-providers-docker    # DockerOperator

# Step 5: Install your scientific packages in the same environment
pip install rdkit requests pandas numpy matplotlib

# Step 6: Register as a Jupyter kernel (for VS Code notebooks)
pip install ipykernel
python -m ipykernel install --user --name=airflow --display-name="Python (airflow)"

# Step 7: Initialise Airflow and create admin user
airflow db init
airflow users create \
    --username admin --password admin \
    --firstname Admin --lastname User \
    --role Admin --email admin@example.com

# Step 8: Start in two terminals (both with venv activated)
airflow webserver --port 8080    # terminal 1
airflow scheduler                # terminal 2

# Open: http://localhost:8080  (admin / admin)
```

### Directory structure

```
~/airflow/
  airflow.cfg          <-- config (database URL, executor, etc.)
  airflow.db           <-- SQLite database (dev only)
  dags/                <-- PUT YOUR DAG FILES HERE
    my_first_dag.py
    toxicology_pipeline.py
  logs/                <-- task execution logs
  plugins/             <-- custom operators
```

### VS Code: select your kernel

```
1. Open Command Palette  Ctrl+Shift+P  (Cmd+Shift+P on Mac)
2. Type: Python: Select Interpreter
3. Choose: Python (airflow)  -- the venv you just created
   Or browse to:  ~/envs/airflow/bin/python
```

In [ ]:
# ── A1: Write and explain your first DAG ────────────────────────────────────
import os

os.makedirs('airflow_dags', exist_ok=True)

first_dag = '''
# ============================================================
# FILE: airflow_dags/my_first_dag.py
# Place this file in ~/airflow/dags/
# ============================================================

from datetime import datetime, timedelta
from airflow.decorators import dag, task

# ── Step 1: Define the DAG with @dag decorator ─────────────────────────────
@dag(
    dag_id          = 'my_first_dag',          # unique name (shown in UI)
    description     = 'A simple hello world DAG',
    start_date      = datetime(2024, 1, 1),     # backfill start date
    schedule        = None,                     # None = manual trigger only
    catchup         = False,                    # don't run missed intervals
    tags            = ['tutorial', 'example'],  # UI labels
    default_args    = {
        'retries'         : 2,                  # retry twice on failure
        'retry_delay'     : timedelta(minutes=5),
        'email_on_failure': False,
    },
)
def my_first_dag():

    # ── Step 2: Define tasks with @task decorator ─────────────────────────
    @task
    def greet():
        print('Hello from Airflow!')
        return 'world'                          # return value passed to next task

    @task
    def process(name: str):                     # receives output of greet()
        message = f'Processing: {name}'
        print(message)
        return message

    @task
    def report(result: str):
        print(f'Pipeline complete! Result: {result}')

    # ── Step 3: Define execution order ────────────────────────────────────
    # greet --> process --> report
    # XCom (cross-communication) automatically passes return values
    greeting = greet()
    processed = process(greeting)
    report(processed)

# ── Step 4: Instantiate the DAG ───────────────────────────────────────────
# This line is required -- it creates the DAG object Airflow scans for
my_first_dag()
'''

with open('airflow_dags/my_first_dag.py', 'w') as f:
    f.write(first_dag)

print('Saved: airflow_dags/my_first_dag.py')
print()
print('To use:')
print('  cp airflow_dags/my_first_dag.py ~/airflow/dags/')
print('  airflow dags list                          # verify it appears')
print('  airflow dags trigger my_first_dag          # trigger manually')
print('  airflow dags test my_first_dag 2024-01-01  # dry-run test')
print()
print('Or use the web UI at http://localhost:8080')

---
## A2: Real Example — Toxicology Data Pipeline

This DAG fetches compound data from PubChem, computes ADMET properties,
screens for structural alerts, and saves a report. A realistic use case
for automating daily compound triage in a drug discovery team.

### Pipeline structure

```
fetch_compounds
       |
       v
compute_admet_properties
       |
    ---+---
    |     |
    v     v
screen_  flag_
alerts   herg
    |     |
    +--+--+
       |
       v
generate_report
       |
       v
send_summary    (email / Slack)
```

This uses the **TaskFlow API** (recommended for modern Airflow 2.x)
where tasks are just decorated Python functions and data flows automatically
between them via **XCom** (cross-task communication).

In [ ]:
tox_dag = '''
# ============================================================
# FILE: airflow_dags/toxicology_pipeline.py
# A real-world compound triage pipeline using Airflow
# ============================================================

from datetime import datetime, timedelta
import json
import requests

from airflow.decorators import dag, task
from airflow.models import Variable


# ── Default arguments applied to every task in this DAG ─────────────────────
DEFAULT_ARGS = {
    'owner'           : 'himanshu',
    'retries'         : 3,
    'retry_delay'     : timedelta(minutes=2),
    'email_on_failure': True,
    'email'           : ['himanshu@example.com'],
}

# ── DAG definition ───────────────────────────────────────────────────────────
@dag(
    dag_id      = 'toxicology_compound_triage',
    description = 'Daily compound triage: fetch, ADMET, alerts, report',
    start_date  = datetime(2024, 1, 1),
    schedule    = '0 6 * * 1-5',   # 6 AM Monday-Friday (cron syntax)
    catchup     = False,
    tags        = ['toxicology', 'admet', 'daily'],
    default_args = DEFAULT_ARGS,
)
def toxicology_pipeline():

    @task
    def fetch_compounds() -> list:
        """
        Step 1: Fetch a list of compounds to triage.
        In production: pull from your compound registry database.
        Here: use PubChem REST API to get a few known drugs.
        """
        compound_names = [
            'aspirin', 'diclofenac', 'caffeine',
            'cisapride', 'metformin', 'tamoxifen'
        ]
        compounds = []
        for name in compound_names:
            url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/IsomericSMILES,MolecularWeight,IUPACName/JSON'
            try:
                r = requests.get(url, timeout=10)
                r.raise_for_status()
                props = r.json()['PropertyTable']['Properties'][0]
                compounds.append({
                    'name'   : name,
                    'smiles' : props.get('IsomericSMILES', ''),
                    'mw'     : props.get('MolecularWeight', 0),
                    'iupac'  : props.get('IUPACName', name),
                    'cid'    : props.get('CID', 0),
                })
                print(f'  Fetched: {name} (CID={props.get("CID", 0)})')
            except Exception as e:
                print(f'  Failed to fetch {name}: {e}')
        print(f'Fetched {len(compounds)} compounds')
        return compounds

    @task
    def compute_admet(compounds: list) -> list:
        """
        Step 2: Compute ADMET properties using RDKit.
        Each compound gets MW, LogP, TPSA, HBD, HBA, QED, Ro5 check.
        """
        from rdkit import Chem
        from rdkit.Chem import Descriptors, rdMolDescriptors, QED

        enriched = []
        for cpd in compounds:
            smiles = cpd.get('smiles', '')
            mol    = Chem.MolFromSmiles(smiles) if smiles else None
            if mol:
                mw    = Descriptors.MolWt(mol)
                logp  = round(Descriptors.MolLogP(mol), 2)
                tpsa  = round(Descriptors.TPSA(mol), 1)
                hbd   = rdMolDescriptors.CalcNumHBD(mol)
                hba   = rdMolDescriptors.CalcNumHBA(mol)
                qed   = round(QED.qed(mol), 3)
                ro5   = sum([mw>500, logp>5, hbd>5, hba>10])
                oral_ok = ro5 <= 1
            else:
                logp = tpsa = hbd = hba = qed = ro5 = None
                oral_ok = False

            enriched.append({
                **cpd,
                'logp'   : logp,
                'tpsa'   : tpsa,
                'hbd'    : hbd,
                'hba'    : hba,
                'qed'    : qed,
                'ro5_violations': ro5,
                'oral_ok': oral_ok,
            })
            print(f'  {cpd["name"]:15s}: LogP={logp}  TPSA={tpsa}  QED={qed}  Oral={oral_ok}')
        return enriched

    @task
    def screen_structural_alerts(compounds: list) -> list:
        """
        Step 3a: Screen for ICH M7 structural alerts (genotoxicity flags).
        Uses RDKit SMARTS matching for 8 key alert patterns.
        """
        from rdkit import Chem

        ALERTS = {
            'Nitrosamine'    : '[N;!$(N=O)]-N=O',
            'Aromatic_nitro' : 'c[N+](=O)[O-]',
            'Aromatic_amine' : '[NH2]c',
            'Michael_acceptor': '[$(C=CC=O)]',
            'Epoxide'        : '[C;R0]1OC1',
            'Hydrazine'      : '[NH2]N',
            'Diazonium'      : '[#6][N+]#N',
            'Furan'          : 'c1ccoc1',
        }

        for cpd in compounds:
            mol    = Chem.MolFromSmiles(cpd.get('smiles','') or '')
            hits   = []
            if mol:
                for alert_name, smarts in ALERTS.items():
                    patt = Chem.MolFromSmarts(smarts)
                    if patt and mol.HasSubstructMatch(patt):
                        hits.append(alert_name)
            c1 = [h for h in hits if 'Nitrosamine' in h]
            cpd['sa_alerts']    = hits
            cpd['n_alerts']     = len(hits)
            cpd['ich_m7_class'] = ('Class 1' if c1 else
                                   'Class 2' if hits else 'Class 5')
            cpd['genotox_flag'] = len(hits) > 0
            if hits:
                print(f'  ALERT {cpd["name"]:12s}: {hits}')
        return compounds

    @task
    def screen_herg(compounds: list) -> list:
        """
        Step 3b: Quick hERG cardiac risk screen (runs in parallel with SA screen).
        Uses lipophilicity + basic nitrogen count as Tier 1 proxy.
        """
        from rdkit import Chem
        from rdkit.Chem import Descriptors, rdMolDescriptors

        for cpd in compounds:
            mol  = Chem.MolFromSmiles(cpd.get('smiles','') or '')
            if mol:
                logp  = Descriptors.MolLogP(mol)
                basic = sum(1 for a in mol.GetAtoms()
                            if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
                naro  = rdMolDescriptors.CalcNumAromaticRings(mol)
                score = 0.3*max(0,logp-2) + 0.15*basic + 0.1*naro
                risk  = 'HIGH' if score>2 else 'MEDIUM' if score>1 else 'LOW'
            else:
                score, risk = 0, 'UNKNOWN'
            cpd['herg_score'] = round(score, 2)
            cpd['herg_risk']  = risk
            if risk == 'HIGH':
                print(f'  hERG HIGH {cpd["name"]:12s}: score={score:.2f}')
        return compounds

    @task
    def merge_results(sa_results: list, herg_results: list) -> list:
        """
        Step 4: Merge the two parallel screen results back into one list.
        Combines SA alerts + hERG risk per compound.
        """
        # Build lookup by compound name
        herg_lookup = {c['name']: c for c in herg_results}
        merged = []
        for cpd in sa_results:
            herg_data = herg_lookup.get(cpd['name'], {})
            merged.append({
                **cpd,
                'herg_score': herg_data.get('herg_score', 0),
                'herg_risk' : herg_data.get('herg_risk', 'UNKNOWN'),
            })
        return merged

    @task
    def generate_report(compounds: list) -> str:
        """
        Step 5: Generate a structured triage report (CSV + summary).
        In production: write to S3, database, or email attachment.
        """
        import pandas as pd
        from datetime import date

        df = pd.DataFrame(compounds)
        today = date.today().isoformat()
        report_path = f'/tmp/tox_triage_{today}.csv'

        # Select key columns for report
        cols = ['name','smiles','mw','logp','tpsa','qed',
                'ro5_violations','oral_ok','n_alerts',
                'ich_m7_class','genotox_flag','herg_risk']
        available = [c for c in cols if c in df.columns]
        df[available].to_csv(report_path, index=False)

        # Summary statistics
        n_total      = len(df)
        n_oral_ok    = df['oral_ok'].sum() if 'oral_ok' in df else 0
        n_genotox    = df['genotox_flag'].sum() if 'genotox_flag' in df else 0
        n_herg_high  = (df['herg_risk']=='HIGH').sum() if 'herg_risk' in df else 0
        n_clean      = sum(1 for _, r in df.iterrows()
                           if r.get('oral_ok') and not r.get('genotox_flag')
                           and r.get('herg_risk') != 'HIGH')

        summary = (
            f'=== Toxicology Triage Report {today} ===\n'
            f'Total compounds:    {n_total}\n'
            f'Oral BA likely:     {n_oral_ok} ({n_oral_ok/n_total*100:.0f}%)\n'
            f'Genotox flags:      {n_genotox}\n'
            f'hERG HIGH risk:     {n_herg_high}\n'
            f'Clean (no flags):   {n_clean}\n'
            f'Report saved:       {report_path}'
        )
        print(summary)
        return report_path

    @task
    def send_summary(report_path: str):
        """
        Step 6: Send Slack/email notification with report path.
        In production: use SlackWebhookOperator or EmailOperator.
        """
        print(f'Notification: Triage complete. Report at {report_path}')
        # Real implementation:
        # hook = SlackWebhookHook(slack_webhook_conn_id='slack_default')
        # hook.send(text=f'Daily tox triage done. Report: {report_path}')

    # ── Wire up the pipeline ─────────────────────────────────────────────────
    raw          = fetch_compounds()
    with_admet   = compute_admet(raw)

    # screen_structural_alerts and screen_herg run IN PARALLEL
    sa_data      = screen_structural_alerts(with_admet)
    herg_data    = screen_herg(with_admet)

    # merge results from both parallel branches
    merged       = merge_results(sa_data, herg_data)
    report       = generate_report(merged)
    send_summary(report)


toxicology_pipeline()
'''

with open('airflow_dags/toxicology_pipeline.py', 'w') as f:
    f.write(tox_dag)
print('Saved: airflow_dags/toxicology_pipeline.py')
print()
print('Pipeline steps:')
steps = [
    ('fetch_compounds',           'PubChem API -> list of SMILES'),
    ('compute_admet',             'RDKit -> MW, LogP, TPSA, QED, Ro5'),
    ('screen_structural_alerts',  'ICH M7 SMARTS -> genotox flags  [PARALLEL]'),
    ('screen_herg',               'LogP + basic N -> cardiac risk   [PARALLEL]'),
    ('merge_results',             'Combine both screens'),
    ('generate_report',           'CSV report to /tmp/'),
    ('send_summary',              'Slack/email notification'),
]
for step, desc in steps:
    print(f'  {step:30s}: {desc}')

---
## A3: Scheduling, Sensors, and Built-in Operators

Airflow has many built-in **Operators** for common tasks, and **Sensors**
that wait for a condition before proceeding.

### Cron schedule syntax

```
schedule = '0 6 * * 1-5'    # 6 AM Mon-Fri
           |  | |  |  |_____ day of week (0=Sun, 1=Mon, 5=Fri)
           |  | |  |________ month (1-12)
           |  | |___________ day of month (1-31)
           |  |_____________ hour (0-23)
           |________________ minute (0-59)

Common examples:
  '@daily'       = '0 0 * * *'     midnight every day
  '@hourly'      = '0 * * * *'     every hour
  '@weekly'      = '0 0 * * 0'     midnight every Sunday
  None                             manual trigger only
```

### Key built-in operators

| Operator | What it does | Example |
|----------|-------------|--------|
| `PythonOperator` | Runs a Python function | compute_descriptors() |
| `BashOperator` | Runs a bash command | `vina --receptor rec.pdbqt ...` |
| `HttpSensor` | Waits for a URL to return 200 | Wait for API to be ready |
| `FileSensor` | Waits for a file to exist | Wait for input SDF file |
| `EmailOperator` | Sends an email | Alert on completion |
| `SlackWebhookOperator` | Sends Slack message | Team notification |
| `DockerOperator` | Runs a Docker container | Run GROMACS in container |

In [ ]:
# ── A3: Key Airflow patterns with code examples ───────────────────────────
import os

# ── Pattern 1: BashOperator (run external tools) ──────────────────────────
bash_example = '''
from airflow.operators.bash import BashOperator

# Run AutoDock Vina via bash inside a DAG
dock_task = BashOperator(
    task_id = 'run_vina_docking',
    bash_command = (
        'vina '
        '--receptor {{ params.receptor }} '
        '--ligand   {{ params.ligand }} '
        '--center_x 2.0 --center_y 5.0 --center_z -10.0 '
        '--size_x 25  --size_y 25  --size_z 25 '
        '--out results/{{ params.name }}_out.pdbqt '
    ),
    params = {
        'receptor': 'receptor/1HVR_receptor.pdbqt',
        'ligand'  : 'ligands/compound_A.pdbqt',
        'name'    : 'compound_A',
    },
)
'''

# ── Pattern 2: FileSensor (wait for input data) ───────────────────────────
sensor_example = '''
from airflow.sensors.filesystem import FileSensor

# Wait for today\'s compound batch file to arrive before processing
wait_for_batch = FileSensor(
    task_id        = 'wait_for_compound_batch',
    filepath       = '/data/incoming/compounds_{{ ds }}.sdf',
    poke_interval  = 60,    # check every 60 seconds
    timeout        = 3600,  # give up after 1 hour
    mode           = 'poke',
)

# Only runs after the file exists
wait_for_batch >> process_compounds_task
'''

# ── Pattern 3: Branching (conditional logic) ──────────────────────────────
branch_example = '''
from airflow.operators.python import BranchPythonOperator

def decide_pathway(ti):
    # Read the compound count from XCom
    count = ti.xcom_pull(task_ids='count_compounds')
    if count > 100:
        return 'run_hpc_job'      # large batch: send to HPC
    else:
        return 'run_local_job'    # small batch: run locally

branch = BranchPythonOperator(
    task_id        = 'choose_compute_resource',
    python_callable = decide_pathway,
)

branch >> [run_hpc_job, run_local_job]
'''

# ── Pattern 4: TaskGroup (organise complex DAGs visually) ─────────────────
taskgroup_example = '''
from airflow.utils.task_group import TaskGroup

with TaskGroup('preparation') as prep_group:
    download = download_structures_task()
    clean    = clean_structures_task(download)

with TaskGroup('screening') as screen_group:
    admet  = run_admet_task()
    alerts = run_sa_screen_task()

with TaskGroup('reporting') as report_group:
    report = generate_report_task()
    notify = send_notification_task(report)

# Groups show as collapsed boxes in the Airflow UI
prep_group >> screen_group >> report_group
'''

# ── Pattern 5: Variables and Connections (config management) ──────────────
variables_example = '''
from airflow.models import Variable
from airflow.hooks.base import BaseHook

# Set via UI: Admin > Variables
compound_batch_size = Variable.get('batch_size', default_var=50)
pubchem_api_key     = Variable.get('pubchem_api_key', deserialize_json=False)

# Set via UI: Admin > Connections
db_conn = BaseHook.get_connection('toxicology_db')
print(f'DB host: {db_conn.host}')
'''

# Print all examples
for name, ex in [
    ('BashOperator',  bash_example),
    ('FileSensor',    sensor_example),
    ('Branching',     branch_example),
    ('TaskGroup',     taskgroup_example),
    ('Variables',     variables_example),
]:
    print(f'Pattern: {name}')
    print(ex)
    print()

---
# PART B — Nextflow

## B1: Installation and Your First Workflow

### Core concepts in 60 seconds

```
Process
  = one computational step (like a function)
  = has: input, output, script (bash, python, R)

Channel
  = a stream of data flowing between processes
  = like a Unix pipe, but asynchronous

Workflow
  = wires processes together via channels
  = the main section of your .nf file

Example:
  Channel(SMILES list)  -->  [PREPARE_LIGAND]  -->  [RUN_DOCKING]
                                                           |
                                                     Channel(scores)
                                                           |
                                                     [RANK_RESULTS]
```

### Installation (pip only where possible)

Nextflow is a Java application, not a Python package.
There is no pip wheel for it. Install via the official installer:

```bash
# Step 1: Make sure Java 11+ is installed
java -version      # must say 11 or higher

# If not installed:
sudo apt install -y default-jdk        # Ubuntu/Debian
brew install openjdk@17               # macOS

# Step 2: Install Nextflow (official one-liner installer)
curl -s https://get.nextflow.io | bash
mv nextflow ~/bin/            # move to somewhere on your PATH
chmod +x ~/bin/nextflow

# Step 3: Verify
nextflow -version

# Step 4: Create a pip venv for Python helper scripts used inside processes
python -m venv ~/envs/nextflow
source ~/envs/nextflow/bin/activate
pip install --upgrade pip
pip install rdkit meeko numpy pandas matplotlib scipy

# Step 5: Register as a Jupyter kernel for VS Code
pip install ipykernel
python -m ipykernel install --user --name=nextflow --display-name="Python (nextflow)"
```

### Why no pip for Nextflow itself?

Nextflow is a JVM application (Groovy/Java), not Python.
The `curl` installer is the official method and is equivalent to
`pip install` for Python — one command, downloads the binary, done.
Your Python packages (RDKit, meeko, etc.) used inside pipeline
processes **do** use pip, in the venv above.

### File structure

```
my_pipeline/
  main.nf             <-- main workflow file
  nextflow.config     <-- configuration (CPUs, memory, Docker)
  bin/                <-- helper scripts callable from processes
  modules/            <-- reusable process definitions
  work/               <-- Nextflow working directory (auto-created)
  results/            <-- your output files
  .nextflow.log       <-- execution log
```

In [ ]:
import os
os.makedirs('nextflow_pipelines', exist_ok=True)

nf_hello = '// ============================================================\n// FILE: nextflow_pipelines/hello_world.nf\n// Run: nextflow run nextflow_pipelines/hello_world.nf\n// ============================================================\n\nnextflow.enable.dsl=2\n\nprocess SAY_HELLO {\n    cpus 1\n    memory \'512 MB\'\n\n    input:\n    val name\n\n    output:\n    stdout\n\n    script:\n    """\n    echo "Hello ${name} from Nextflow!"\n    echo "Running on: $(hostname)"\n    """\n}\n\nprocess COUNT_LETTERS {\n    input:\n    val message\n\n    output:\n    stdout\n\n    script:\n    """\n    echo "${message}" | wc -c\n    """\n}\n\nworkflow {\n    names_ch = Channel.of( \'Alice\', \'Bob\', \'Charlie\' )\n    greetings_ch = SAY_HELLO(names_ch)\n    greetings_ch.view { "Greeting: $it" }\n    COUNT_LETTERS(greetings_ch).view { it.trim() }\n}'

with open('nextflow_pipelines/hello_world.nf', 'w') as f:
    f.write(nf_hello)
print('Saved: nextflow_pipelines/hello_world.nf')
print()
print('To run:')
print('  nextflow run nextflow_pipelines/hello_world.nf')
print()
print('Key concepts demonstrated:')
concepts = [
    'Channel.of() -- creates a channel from a list',
    'SAY_HELLO runs 3 times IN PARALLEL (one per name)',
    'Output of SAY_HELLO flows into COUNT_LETTERS automatically',
    '.view{} -- prints channel contents for debugging',
    'Each task gets its own isolated work/ subdirectory',
]
for c in concepts:
    print(f'  * {c}')

In [ ]:
// nextflow.config -- global configuration
nextflow_config_content = '''
// ============================================================
// FILE: nextflow_pipelines/nextflow.config
// ============================================================

// Default process settings
process {
    cpus   = 2
    memory = '4 GB'
    time   = '2h'
}

// Profile: local (default)
profiles {
    local {
        process.executor = 'local'
    }
    // Profile: SLURM HPC
    slurm {
        process.executor = 'slurm'
        process.queue    = 'normal'
        process {
            withLabel: 'big_job' {
                cpus   = 16
                memory = '64 GB'
                time   = '24h'
            }
        }
    }
    // Profile: Docker containers
    docker {
        docker.enabled = true
        process.container = 'biocontainers/rdkit:latest'
    }
}

// Output directory
params.outdir = 'results'
'''

with open('nextflow_pipelines/nextflow.config', 'w') as f:
    f.write(nextflow_config_content)
print('Saved: nextflow_pipelines/nextflow.config')
print()
print('Run with different profiles:')
print('  nextflow run main.nf -profile local')
print('  nextflow run main.nf -profile slurm')
print('  nextflow run main.nf -profile docker')

---
## B2: Real Example — Molecular Docking Screening Pipeline

This Nextflow pipeline takes a CSV of SMILES, prepares each ligand,
docks it against HIV-1 Protease, and produces a ranked results table.

### Why Nextflow is perfect for this

- Naturally parallel: each compound docks independently
- Scatter-gather pattern: fan out 100 compounds, fan back in to one results file
- Resumable: if docking fails on compound 47, `nextflow run -resume` skips the other 99
- Portable: run locally, on SLURM, or in Docker with one flag change

### Pipeline structure

```
Input: compounds.csv
       |
       | splitCsv() -- one row per item in channel
       v
PREPARE_LIGAND  x100 (parallel)
       |
       v
RUN_DOCKING     x100 (parallel)
       |
       | collect() -- gather all results
       v
PARSE_SCORES    (single task)
       |
       v
RANK_AND_REPORT (single task)
Output: results/ranked_hits.csv
```

In [ ]:
docking_nf = 'nextflow.enable.dsl=2\n\n// ── Parameters (override on command line: --receptor path/to/rec.pdbqt) ─────\nparams.compounds_csv = \'compounds.csv\'\nparams.receptor      = \'receptor/1HVR_receptor.pdbqt\'\nparams.center_x      = 2.0\nparams.center_y      = 5.0\nparams.center_z      = -10.0\nparams.size          = 25\nparams.exhaustiveness = 8\nparams.outdir        = \'results\'\n\n// ── Process 1: Prepare ligand (SMILES -> PDBQT) ───────────────────────────\nprocess PREPARE_LIGAND {\n    tag "${name}"      // shown in logs and UI\n    publishDir "${params.outdir}/ligands", mode: \'copy\'\n\n    input:\n    tuple val(name), val(smiles)\n\n    output:\n    tuple val(name), path("${name}.pdbqt"), optional: true\n\n    script:\n    """\n    python3 - << \'PYEOF\'\nimport sys\nfrom rdkit import Chem\nfrom rdkit.Chem import AllChem\nfrom meeko import MoleculePreparation, PDBQTWriterLegacy\n\nname   = \'${name}\'\nsmiles = \'${smiles}\'\nmol    = Chem.MolFromSmiles(smiles)\nif mol is None:\n    print(f\'Invalid SMILES for {name}\', file=sys.stderr)\n    sys.exit(0)\n\nmol = Chem.AddHs(mol)\nAllChem.EmbedMolecule(mol, AllChem.ETKDGv3())\nAllChem.MMFFOptimizeMolecule(mol)\n\nprep   = MoleculePreparation()\nsetups = prep.prepare(mol)\npdbqt, ok, err = PDBQTWriterLegacy.write_string(setups[0])\nif ok:\n    with open(f\'{name}.pdbqt\', \'w\') as f:\n        f.write(pdbqt)\n    print(f\'Prepared {name}\')\nPYEOF\n    """\n}\n\n// ── Process 2: Run AutoDock Vina ──────────────────────────────────────────\nprocess RUN_DOCKING {\n    tag "${name}"\n    publishDir "${params.outdir}/poses", mode: \'copy\'\n    cpus 4\n\n    input:\n    tuple val(name), path(ligand_pdbqt)\n\n    output:\n    tuple val(name), path("${name}_out.pdbqt"), path("${name}_score.txt")\n\n    script:\n    """\n    vina \\\n        --receptor ${params.receptor} \\\n        --ligand   ${ligand_pdbqt} \\\n        --center_x ${params.center_x} \\\n        --center_y ${params.center_y} \\\n        --center_z ${params.center_z} \\\n        --size_x   ${params.size} \\\n        --size_y   ${params.size} \\\n        --size_z   ${params.size} \\\n        --exhaustiveness ${params.exhaustiveness} \\\n        --num_modes 9 \\\n        --out ${name}_out.pdbqt \\\n        --cpu ${task.cpus} 2>&1 | tee vina_log.txt\n\n    # Extract best score from Vina output\n    grep -A 20 \'mode |\' vina_log.txt | grep \'^ *1 \' | awk \'{print "${name}," $2}\' > ${name}_score.txt || echo \'${name},NA\' > ${name}_score.txt\n    """\n}\n\n// ── Process 3: Collect and rank all results ───────────────────────────────\nprocess RANK_RESULTS {\n    publishDir params.outdir, mode: \'copy\'\n\n    input:\n    path score_files\n\n    output:\n    path \'ranked_hits.csv\'\n\n    script:\n    """\n    python3 - << \'PYEOF\'\nimport glob, pandas as pd\n\nrecords = []\nfor f in glob.glob(\'*_score.txt\'):\n    with open(f) as fh:\n        line = fh.read().strip()\n        if \',\' in line:\n            name, score = line.split(\',\', 1)\n            try:\n                records.append({\'name\': name.strip(), \'affinity_kcal_mol\': float(score.strip())})\n            except ValueError:\n                records.append({\'name\': name.strip(), \'affinity_kcal_mol\': None})\n\ndf = pd.DataFrame(records).dropna()\ndf = df.sort_values(\'affinity_kcal_mol\').reset_index(drop=True)\ndf[\'rank\'] = range(1, len(df)+1)\ndf.to_csv(\'ranked_hits.csv\', index=False)\nprint(df.to_string(index=False))\nPYEOF\n    """\n}\n\n// ── Workflow: wire processes together ─────────────────────────────────────\nworkflow {\n\n    // Read the CSV: each row becomes one item in the channel\n    // CSV format: name,smiles\n    compounds_ch = Channel\n        .fromPath(params.compounds_csv)\n        .splitCsv(header: true)\n        .map { row -> tuple(row.name, row.smiles) }\n\n    // Prepare all ligands in parallel\n    ligands_ch = PREPARE_LIGAND(compounds_ch)\n\n    // Dock each ligand in parallel (100 jobs run simultaneously)\n    docked_ch  = RUN_DOCKING(ligands_ch)\n\n    // Collect all score files, then rank in one task\n    score_files_ch = docked_ch.map { name, pdbqt, score -> score }.collect()\n    RANK_RESULTS(score_files_ch)\n}'

with open('nextflow_pipelines/docking_pipeline.nf', 'w') as f:
    f.write(docking_nf)
print('Saved: nextflow_pipelines/docking_pipeline.nf')

# Also create a sample compounds.csv
import pandas as pd
compounds = pd.DataFrame([
    {'name':'Indinavir',   'smiles':'Cc1nc2n(c1=O)CC(N1CC(O)CC1Cc1ccccc1)C(=O)Nc1cc(ccc1)C(=O)NC2'},
    {'name':'Saquinavir',  'smiles':'CC(C)(C)NC(=O)C1CC2CCCCC2CN1CC(O)C(Cc1ccccc1)NC(=O)C(CC(N)=O)NC(=O)c1ccc2ccccc2n1'},
    {'name':'Compound_A',  'smiles':'O=C(Nc1cccc(F)c1)C1CCN(Cc2ccc(Cl)cc2)CC1'},
    {'name':'Aspirin',     'smiles':'CC(=O)Oc1ccccc1C(=O)O'},
    {'name':'Caffeine',    'smiles':'Cn1cnc2c1c(=O)n(C)c(=O)n2C'},
])
compounds.to_csv('nextflow_pipelines/compounds.csv', index=False)
print('Saved: nextflow_pipelines/compounds.csv')
print()
print('To run:')
print('  cd nextflow_pipelines')
print('  nextflow run docking_pipeline.nf')
print()
print('Useful run flags:')
flags = [
    ('-resume',                   'Restart from last successful step (saves time!)'),
    ('-with-report report.html',  'Generate beautiful HTML execution report'),
    ('-with-timeline timeline.html','Show task execution timeline'),
    ('-with-dag dag.png',          'Draw the pipeline DAG'),
    ('-profile docker',            'Run everything in Docker containers'),
    ('-profile slurm',             'Submit to SLURM HPC queue'),
    ('--outdir my_results/',       'Override output directory'),
]
for flag, desc in flags:
    print(f'  {flag:35s}: {desc}')

---
## B3: Docker Containers and HPC Submission

Nextflow's biggest advantage is **portability**. The same `.nf` file runs
locally, in Docker, on SLURM, on AWS, or on Google Cloud by changing
one `-profile` flag.

### Docker containers make science reproducible

```
Without Docker              With Docker
--------------              -----------
'It works on my machine'    Same container everywhere
Dependency hell             Pinned versions
Hard to share               Share image on Docker Hub
Cluster dependencies vary   Self-contained environment
```

### Nextflow + Docker: how it works

```
Each process runs in its own Docker container.
Nextflow mounts the work/ directory automatically.
You just specify: process.container = 'image:tag'
```

### SLURM integration: one flag

```bash
# Local (default)
nextflow run main.nf

# SLURM HPC -- submits each process as a SLURM job
nextflow run main.nf -profile slurm

# AWS Batch
nextflow run main.nf -profile aws

# All of the above, with Docker
nextflow run main.nf -profile slurm,docker
```

In [ ]:
# ── B3: Advanced nextflow.config with Docker + SLURM profiles ───────────────

advanced_config_lines = [
    'nextflow.enable.dsl=2',
    '',
    '// ============================================================',
    '// nextflow_pipelines/nextflow_advanced.config',
    '// ============================================================',
    '',
    '// Global parameters',
    'params {',
    "    outdir        = 'results'",
    "    receptor      = 'receptor/1HVR_receptor.pdbqt'",
    "    compounds_csv = 'compounds.csv'",
    "    exhaustiveness = 8",
    '}',
    '',
    '// Default process resources',
    'process {',
    "    cpus   = 2",
    "    memory = '8 GB'",
    "    time   = '2h'",
    '',
    '    // Label-based resource overrides',
    '    withLabel: gpu_task {',
    "        cpus        = 8",
    "        memory      = '32 GB'",
    "        time        = '24h'",
    "        clusterOptions = '--gres=gpu:1'",
    '    }',
    '',
    '    withLabel: quick_task {',
    "        cpus   = 1",
    "        memory = '2 GB'",
    "        time   = '30m'",
    '    }',
    '}',
    '',
    '// Profiles',
    'profiles {',
    '',
    '    local {',
    "        process.executor = 'local'",
    '    }',
    '',
    '    // Docker: each process runs in its own container',
    '    docker {',
    '        docker.enabled   = true',
    '        docker.runOptions = \'--user $(id -u):$(id -g)\'',
    '        process {',
    "            withName: PREPARE_LIGAND { container = 'mcs07/rdkit:latest' }",
    "            withName: RUN_DOCKING    { container = 'informatics/autodock-vina:latest' }",
    "            withName: RANK_RESULTS   { container = 'python:3.10-slim' }",
    '        }',
    '    }',
    '',
    '    // Singularity: use on HPC (no root required)',
    '    singularity {',
    '        singularity.enabled    = true',
    '        singularity.autoMounts = true',
    '        process {',
    "            withName: RUN_DOCKING { container = 'docker://informatics/autodock-vina:latest' }",
    '        }',
    '    }',
    '',
    '    // SLURM HPC cluster',
    '    slurm {',
    "        process.executor    = 'slurm'",
    "        process.queue       = 'normal'",
    "        process.clusterOptions = '--account=my_project'",
    '        executor {',
    '            queueSize         = 200  // max concurrent SLURM jobs',
    '            submitRateLimit   = \'10/1min\'',
    '        }',
    '    }',
    '',
    '    // AWS Batch',
    '    aws {',
    "        process.executor    = 'awsbatch'",
    "        process.queue       = 'nextflow-batch-queue'",
    "        aws.region          = 'us-east-1'",
    "        workDir             = 's3://my-bucket/nextflow-work'",
    '    }',
    '}',
]

config_content = '\n'.join(advanced_config_lines)
with open('nextflow_pipelines/nextflow_advanced.config', 'w') as f:
    f.write(config_content)
print('Saved: nextflow_pipelines/nextflow_advanced.config')

# Show a process with Docker container and label
process_with_docker = '''
// Process with Docker container and resource label
process RUN_DOCKING {
    tag "${name}"
    label 'gpu_task'                          // uses gpu_task resource profile
    container 'informatics/autodock-vina:1.2' // exact Docker image version
    publishDir "${params.outdir}/poses"

    input:
    tuple val(name), path(ligand_pdbqt)

    output:
    tuple val(name), path("${name}_out.pdbqt"), path("${name}_score.txt")

    script:
    // Uses ${task.cpus} so SLURM resource is automatically passed to Vina
    """
    vina --receptor ${params.receptor} --ligand ${ligand_pdbqt} \\
         --center_x ${params.center_x} --size_x ${params.size} \\
         --exhaustiveness ${params.exhaustiveness} --cpu ${task.cpus} \\
         --out ${name}_out.pdbqt 2>&1 | tee vina_log.txt
    """
}
'''
print(process_with_docker)

print('Workflow execution commands:')
commands = [
    ('nextflow run main.nf',                         'Local, no containers'),
    ('nextflow run main.nf -profile docker',          'Local with Docker'),
    ('nextflow run main.nf -profile slurm',           'SLURM HPC'),
    ('nextflow run main.nf -profile slurm,singularity','SLURM + Singularity'),
    ('nextflow run main.nf -resume',                   'Restart from checkpoint'),
    ('nextflow run main.nf -with-report report.html',  'HTML execution report'),
]
for cmd, desc in commands:
    print(f'  {cmd:50s}: {desc}')

---
## Visual Summary: Airflow vs Nextflow Side by Side

This section shows the same pipeline concept written in both tools,
so you can see exactly how the syntax and philosophy differ.

In [ ]:
# ── Side-by-side comparison: same pipeline in Airflow and Nextflow ───────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.55, wspace=0.35)

# Colour scheme
AF_COL  = '#017CEE'  # Airflow blue
NF_COL  = '#3EAA47'  # Nextflow green
BG_DARK = '#1a1a2e'

def draw_pipeline(ax, steps, connections, title, colour, subtitle=''):
    ax.set_xlim(0, 10); ax.set_ylim(0, len(steps)*2+1)
    ax.set_facecolor('#f8f9fa')
    ax.axis('off')
    ax.set_title(f'{title}\n{subtitle}', fontweight='bold', fontsize=12,
                 color=colour, pad=10)
    y_positions = {}
    for i, (step_id, label, sublabel) in enumerate(steps):
        y = len(steps)*2 - i*2
        y_positions[step_id] = y
        rect = mpatches.FancyBboxPatch(
            (1, y-0.55), 8, 1.1,
            boxstyle='round,pad=0.08',
            facecolor=colour, alpha=0.15,
            edgecolor=colour, lw=2)
        ax.add_patch(rect)
        ax.text(5, y+0.15, label, ha='center', va='center',
                fontsize=9.5, fontweight='bold', color='#111')
        ax.text(5, y-0.2, sublabel, ha='center', va='center',
                fontsize=8, color='#555', style='italic')
    for src, dst, label in connections:
        y1 = y_positions[src] - 0.55
        y2 = y_positions[dst] + 0.55
        ax.annotate('', xy=(5, y2), xytext=(5, y1),
                    arrowprops=dict(arrowstyle='->', color=colour, lw=2))
        ymid = (y1+y2)/2
        if label:
            ax.text(5.3, ymid, label, fontsize=7.5, color=colour, style='italic')

# Airflow pipeline (DAG view)
ax1 = fig.add_subplot(gs[0, 0])
af_steps = [
    ('fetch',   'fetch_compounds()',         '@task -- calls PubChem API'),
    ('admet',   'compute_admet()',            '@task -- RDKit descriptors'),
    ('sa',      'screen_structural_alerts()', '@task -- SMARTS patterns'),
    ('herg',    'screen_herg()',              '@task -- parallel with SA'),
    ('merge',   'merge_results()',            '@task -- joins parallel branches'),
    ('report',  'generate_report()',          '@task -- CSV + summary'),
    ('notify',  'send_summary()',             '@task -- Slack/email'),
]
af_conns = [
    ('fetch','admet','XCom: list[dict]'),
    ('admet','sa','XCom: list[dict]'),
    ('admet','herg','XCom: list[dict]'),
    ('sa','merge','XCom: sa_results'),
    ('herg','merge','XCom: herg_results'),
    ('merge','report','XCom: merged_data'),
    ('report','notify','XCom: report_path'),
]
draw_pipeline(ax1, af_steps, af_conns, 'Apache Airflow', AF_COL, 'Python DAG (TaskFlow API)')

# Nextflow pipeline
ax2 = fig.add_subplot(gs[0, 1])
nf_steps = [
    ('csv',     'Channel.fromPath(csv).splitCsv()', 'Channel of (name, smiles) tuples'),
    ('prep',    'PREPARE_LIGAND',                   'x100 parallel, one per compound'),
    ('dock',    'RUN_DOCKING',                      'x100 parallel, runs vina'),
    ('collect', '.collect()',                       'Gather all score files'),
    ('rank',    'RANK_RESULTS',                     'Single task, produces CSV'),
]
nf_conns = [
    ('csv','prep','Channel: (name, smiles)'),
    ('prep','dock','Channel: (name, pdbqt)'),
    ('dock','collect','Channel: score_files'),
    ('collect','rank','Path[]: all scores'),
]
draw_pipeline(ax2, nf_steps, nf_conns, 'Nextflow', NF_COL, 'DSL2 Workflow')

# Feature comparison table
ax3 = fig.add_subplot(gs[1, :])
ax3.axis('off')
headers = ['Feature', 'Apache Airflow', 'Nextflow', 'When to choose']
rows = [
    ['Language',       'Python (DAGs)',       'Groovy/DSL2',              'Airflow if team knows Python'],
    ['Scheduling',     'Yes (cron-based)',     'No (trigger-based)',        'Airflow for recurring jobs'],
    ['Parallelism',    'Tasks within DAG',    'Native scatter-gather',     'Nextflow for many-sample science'],
    ['Containers',     'Optional',            'First-class Docker/Singularity','Nextflow for reproducibility'],
    ['HPC/SLURM',      'Via BashOperator',    'Native executor',           'Nextflow for compute clusters'],
    ['Cloud',          'Via operators',       'Native (AWS/GCP/Azure)',    'Both work; Nextflow easier'],
    ['Monitoring',     'Web UI + alerts',     'HTML report + Seqera',      'Airflow better for ops teams'],
    ['Resume',         'No (restart from 0)', 'Yes (-resume flag)',         'Nextflow saves compute time'],
    ['Data handling',  'XCom (small data)',   'Channels (any size)',        'Nextflow for large files'],
    ['Best use case',  'ETL, data pipelines', 'Bioinformatics, screening', 'Pick by use case'],
]
tbl = ax3.table(
    cellText   = rows,
    colLabels  = headers,
    cellLoc    = 'left',
    loc        = 'center',
    bbox       = [0, 0, 1, 1]
)
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for j in range(len(headers)):
    tbl[0, j].set_facecolor('#333'); tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(rows)+1):
    for j in range(len(headers)):
        bg = '#EBF5FB' if i%2==0 else 'white'
        tbl[i, j].set_facecolor(bg)
        if j == 1: tbl[i, j].set_text_props(color=AF_COL)
        if j == 2: tbl[i, j].set_text_props(color=NF_COL)
ax3.set_title('Airflow vs Nextflow Feature Comparison', fontweight='bold', fontsize=12, pad=10)

fig.suptitle('Workflow Orchestration: Apache Airflow & Nextflow',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('workflow_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: workflow_comparison.png')

In [ ]:
lines = [
    'AIRFLOW + NEXTFLOW -- PIP-ONLY SETUP & QUICK REFERENCE',
    '',
    'SETUP: AIRFLOW (pip + venv)',
    '  python -m venv ~/envs/airflow',
    '  source ~/envs/airflow/bin/activate',
    '  pip install --upgrade pip',
    '  pip install apache-airflow==2.9.1 \\',
    '    --constraint https://raw.githubusercontent.com/apache/airflow/constraints-2.9.1/constraints-3.10.txt',
    '  pip install rdkit requests pandas numpy matplotlib ipykernel',
    '  python -m ipykernel install --user --name=airflow --display-name="Python (airflow)"',
    '  airflow db init',
    '  airflow users create --username admin --password admin --role Admin ...',
    '  airflow webserver --port 8080   # terminal 1',
    '  airflow scheduler               # terminal 2',
    '',
    'SETUP: NEXTFLOW (curl installer -- no pip, it is a Java app)',
    '  java -version                              # need Java 11+',
    '  curl -s https://get.nextflow.io | bash    # install nextflow binary',
    '  mv nextflow ~/bin/ && chmod +x ~/bin/nextflow',
    '  nextflow -version                          # verify',
    '  # Python packages for use inside Nextflow processes:',
    '  python -m venv ~/envs/nextflow',
    '  source ~/envs/nextflow/bin/activate',
    '  pip install --upgrade pip',
    '  pip install rdkit meeko numpy pandas matplotlib scipy ipykernel',
    '  python -m ipykernel install --user --name=nextflow --display-name="Python (nextflow)"',
    '',
    'AIRFLOW -- KEY SYNTAX',
    '  DAG skeleton:',
    '    @dag(dag_id, schedule, start_date, catchup=False)',
    '    def pipeline():',
    '        @task',
    '        def step_a(): return data',
    '        @task',
    '        def step_b(data): ...',
    '        step_b(step_a())',
    '    pipeline()              # instantiate at module level',
    '',
    '  Dependencies:',
    '    a >> b               a then b',
    '    a >> [b, c]          a then b AND c (parallel)',
    '    [b, c] >> d          both b and c then d',
    '',
    '  Schedule: None=manual  @daily  @hourly  0 6 * * 1-5',
    '',
    '  Key operators:',
    '    BashOperator, FileSensor, HttpSensor, EmailOperator',
    '    BranchPythonOperator, DockerOperator',
    '',
    '  CLI:',
    '    airflow dags list',
    '    airflow dags trigger my_dag',
    '    airflow dags test my_dag 2024-01-01',
    '',
    'NEXTFLOW -- KEY SYNTAX',
    '  Process skeleton:',
    '    process MY_STEP {',
    '        tag "${name}"',
    '        publishDir "results/"',
    '        input:  val name',
    '        output: path "${name}.txt"',
    '        script: ... bash commands ...',
    '    }',
    '',
    '  Channel operations:',
    '    Channel.of(1,2,3)          from values',
    '    Channel.fromPath("*.sdf")  from file glob',
    '    .splitCsv(header:true)     parse CSV rows',
    '    .map { row -> row.name }   transform items',
    '    .collect()                 gather all into list',
    '    .view()                    print channel contents',
    '',
    '  Run commands:',
    '    nextflow run main.nf',
    '    nextflow run main.nf -resume',
    '    nextflow run main.nf -profile docker',
    '    nextflow run main.nf -profile slurm',
    '    nextflow run main.nf -with-report report.html',
    '    nextflow run main.nf -with-dag dag.png',
    '',
    'CHOOSE AIRFLOW WHEN:',
    '  - Scheduled recurring jobs (daily, weekly)',
    '  - Team knows Python',
    '  - Need monitoring dashboard with email/Slack alerts',
    '  - Orchestrating APIs, databases, cloud services',
    '',
    'CHOOSE NEXTFLOW WHEN:',
    '  - Scientific analysis pipeline (many samples)',
    '  - Natural parallelism (one process per file/compound)',
    '  - Need Docker/Singularity for reproducibility',
    '  - Running on HPC cluster (SLURM, PBS)',
    '  - Want -resume after partial failure',
]
print('\n'.join(lines))


In [ ]:
import os

files = [
    ('airflow_dags/my_first_dag.py',             'Airflow hello world DAG'),
    ('airflow_dags/toxicology_pipeline.py',       'Airflow full tox triage pipeline'),
    ('nextflow_pipelines/hello_world.nf',         'Nextflow hello world'),
    ('nextflow_pipelines/nextflow.config',         'Nextflow basic config'),
    ('nextflow_pipelines/nextflow_advanced.config','Nextflow Docker + SLURM + AWS'),
    ('nextflow_pipelines/docking_pipeline.nf',    'Nextflow docking screening pipeline'),
    ('nextflow_pipelines/compounds.csv',           'Sample compound library'),
]

print('Files created:')
for path, desc in files:
    status = 'OK' if os.path.exists(path) else '--'
    print(f'  [{status}] {path:46s} {desc}')

print()
print('Quick start:')
print('  AIRFLOW:  cp airflow_dags/*.py ~/airflow/dags/')
print('            airflow dags trigger toxicology_compound_triage')
print('  NEXTFLOW: cd nextflow_pipelines && nextflow run hello_world.nf')
print('            nextflow run docking_pipeline.nf -with-report report.html')
